# L14 - Il perceptron che impara

Eseguire le celle in ordine con `Maiusc + Invio`.

In [ ]:
# Funzione di controllo degli esercizi: eseguire questa cella, non modificarla.
def controlla(numero, prova):
    try:
        esito = bool(prova())
    except Exception as e:
        print(f"Esercizio {numero}: da rivedere ({type(e).__name__})")
        return
    print(f"Esercizio {numero}: " + ("corretto" if esito else "da rivedere"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

P = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

def neurone(X, w, b):
    return np.where(X @ w + b >= 0, 1, 0)

## 1. La regola di apprendimento

Per ogni esempio $(x, y)$: si calcola la previsione $\hat{y}$ e l'errore $e = y - \hat{y}$ (che vale -1, 0 oppure 1), poi si aggiornano pesi e bias:

$$w \leftarrow w + \eta \, e \, x \qquad b \leftarrow b + \eta \, e$$

$\eta$ (eta) è il tasso di apprendimento. Se la previsione è corretta, $e = 0$ e nulla cambia.

In [ ]:
def addestra(X, y, eta=0.1, epoche=20, seme=0, stampa=False):
    """Addestra un perceptron. Restituisce pesi, bias e numero di errori per epoca."""
    rng = np.random.default_rng(seme)
    w = rng.normal(0, 0.1, size=X.shape[1])    # pesi iniziali piccoli e casuali
    b = 0.0
    errori_per_epoca = []
    for epoca in range(epoche):
        errori = 0
        for x_i, y_i in zip(X, y):
            y_prev = 1 if x_i @ w + b >= 0 else 0
            e = y_i - y_prev
            w = w + eta * e * x_i
            b = b + eta * e
            if e != 0:
                errori += 1
        errori_per_epoca.append(errori)
        if stampa:
            print(f"epoca {epoca + 1:2}: errori {errori}, w = {np.round(w, 3)}, b = {b:.3f}")
    return w, b, errori_per_epoca

w, b, err = addestra(P, np.array([0, 0, 0, 1]), stampa=True)
print("uscite finali:", neurone(P, w, b))

## 2. La curva degli errori

Il numero di errori per epoca mostra l'andamento dell'addestramento. Quando arriva a 0 il perceptron classifica correttamente tutti gli esempi e smette di cambiare.

In [ ]:
plt.plot(range(1, len(err) + 1), err, marker="o")
plt.xlabel("epoca")
plt.ylabel("errori")
plt.title("Perceptron su AND")
plt.show()

## 3. Un problema con più dati

Due classi di punti generate casualmente. Il perceptron cerca una retta che le separi; la cella disegna la retta dopo alcune epoche.

In [ ]:
rng = np.random.default_rng(3)
A = rng.normal(loc=[2, 2], scale=0.6, size=(40, 2))
B = rng.normal(loc=[4.5, 4.5], scale=0.6, size=(40, 2))
X = np.vstack([A, B])                          # 80 righe: prima A, poi B
y = np.array([0] * 40 + [1] * 40)

plt.scatter(A[:, 0], A[:, 1], marker="o", label="classe 0")
plt.scatter(B[:, 0], B[:, 1], marker="s", label="classe 1")
x1 = np.linspace(0, 7, 50)
for ep in [3, 10, 20]:
    w, b, err = addestra(X, y, eta=0.1, epoche=ep)
    plt.plot(x1, -(w[0] * x1 + b) / w[1], label=f"dopo {ep} epoche")
plt.xlim(0, 7)
plt.ylim(0, 7)
plt.legend()
plt.show()

`np.vstack` impila verticalmente due matrici con lo stesso numero di colonne.

## 4. XOR

Su XOR il perceptron non arriva mai a zero errori: nessuna retta separa i punti.

In [ ]:
w, b, err = addestra(P, np.array([0, 1, 1, 0]), epoche=30)
print("errori per epoca:", err)

## Esercizi

**Esercizio 1 (base).** Addestrare il perceptron su OR e verificare le uscite. Assegnare a `w_or`, `b_or` i parametri trovati e a `err_or` la lista degli errori per epoca.

In [ ]:
y_or = np.array([0, 1, 1, 1])
w_or, b_or, err_or = addestra(P, y_or)
print(neurone(P, w_or, b_or), err_or)

In [ ]:
controlla(1, lambda: np.array_equal(neurone(P, w_or, b_or), y_or) and err_or[-1] == 0)

**Esercizio 2 (base).** Per il problema della sezione 3 calcolare `epoca_zero`: il numero della prima epoca (contando da 1) in cui gli errori sono 0, con `eta=0.1`, 30 epoche e seme 0. Suggerimento: `lista.index(valore)` restituisce la posizione del primo elemento uguale a `valore`.

In [ ]:
w, b, err = addestra(X, y, eta=0.1, epoche=30)
epoca_zero = err.index(0) + 1
print(err)

In [ ]:
controlla(2, lambda: epoca_zero == addestra(X, y, eta=0.1, epoche=30)[2].index(0) + 1)

**Esercizio 3 (standard).** Scrivere `accuratezza(X, y, w, b)`: la frazione di esempi classificati correttamente. Usarla per valutare il perceptron dopo 1 epoca e dopo 30 epoche.

In [ ]:
def accuratezza(X, y, w, b):
    return (neurone(X, w, b) == y).mean()

w1_, b1_, _ = addestra(X, y, epoche=1)
w30, b30, _ = addestra(X, y, epoche=30)
print(accuratezza(X, y, w1_, b1_), accuratezza(X, y, w30, b30))

In [ ]:
controlla(3, lambda: accuratezza(P, np.array([0, 0, 0, 1]), np.array([1, 1]), -1.5) == 1.0 and accuratezza(P, np.array([0, 1, 1, 0]), np.array([1, 1]), -1.5) == 0.25)

**Esercizio 4 (standard).** Effetto del tasso di apprendimento. Addestrare sul problema della sezione 3 con `eta` pari a 0.01, 0.1 e 1, per 30 epoche, e disegnare nello stesso grafico le tre curve degli errori. Salvare le tre liste di errori nel dizionario `curve` con chiave il valore di `eta`. Osservare: il valore di `eta` cambia molto il risultato? Perché? (Suggerimento: moltiplicare pesi e bias per lo stesso numero positivo non cambia la retta.)

In [ ]:
curve = {}
for eta in [0.01, 0.1, 1]:
    _, _, e = addestra(X, y, eta=eta, epoche=30)
    curve[eta] = e
    plt.plot(range(1, 31), e, marker=".", label=f"eta = {eta}")
plt.xlabel("epoca")
plt.ylabel("errori")
plt.legend()
plt.show()

In [ ]:
controlla(4, lambda: set(curve.keys()) == {0.01, 0.1, 1} and all(len(v) == 30 for v in curve.values()))

**Esercizio 5 (approfondimento).** Scrivere la classe `Perceptron` con i metodi `fit(X, y)`, che addestra e memorizza pesi e bias negli attributi `w` e `b`, e `predict(X)`, che restituisce le uscite. È la stessa interfaccia delle classi di scikit-learn (lezione L17). Il costruttore riceve `eta` ed `epoche`.

In [ ]:
class Perceptron:
    def __init__(self, eta=0.1, epoche=20):
        self.eta = eta
        self.epoche = epoche

    def fit(self, X, y):
        self.w, self.b, self.errori = addestra(X, y, self.eta, self.epoche)
        return self

    def predict(self, X):
        return neurone(X, self.w, self.b)

In [ ]:
def _prova5():
    m = Perceptron(eta=0.1, epoche=30)
    m.fit(X, y)
    return (m.predict(X) == y).mean() == 1.0 and np.array_equal(Perceptron().fit(P, np.array([0, 0, 0, 1])).predict(P), [0, 0, 0, 1])
controlla(5, _prova5)